In [1]:
import radiate as rd
import polars as pl
from IPython.display import display, HTML

rd.random.seed(33)

In [2]:
inputs, answers = rd.DataSet.cubic().pair()
collector = rd.MetricCollector()

wait = (
    rd.Expr.when(rd.Expr.generation() > 10)
    .then(rd.Expr.select("scores.best").rolling(2).slope())
    .otherwise(0)
)

slope = rd.Expr.select("scores.best").rolling(5).slope() * 0.02

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.sum()],
        edge=rd.Op.weight(),
        output=rd.Op.sum(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .metrics(slope=slope)
    .alter(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-11-15T16:45:50.013532Z  INFO State Change: PreStart -> Running
2026-11-15T16:45:50.014434Z  INFO Epoch 1    | Score:   2.9954 | Time: 698.04µs
2026-11-15T16:45:50.014533Z  INFO Epoch 2    | Score:   2.0038 | Time: 747.71µs
2026-11-15T16:45:50.014590Z  INFO Epoch 3    | Score:   2.0038 | Time: 786.50µs
2026-11-15T16:45:50.014648Z  INFO Epoch 4    | Score:   2.0038 | Time: 832.12µs
2026-11-15T16:45:50.014702Z  INFO Epoch 5    | Score:   2.0038 | Time: 872.83µs
2026-11-15T16:45:50.014777Z  INFO Epoch 6    | Score:   1.6821 | Time: 933.58µs
2026-11-15T16:45:50.014836Z  INFO Epoch 7    | Score:   1.6821 | Time: 981.71µs
2026-11-15T16:45:50.014903Z  INFO Epoch 8    | Score:   1.6821 | Time: 1.04ms
2026-11-15T16:45:50.014963Z  INFO Epoch 9    | Score:   1.6821 | Time: 1.08ms
2026-11-15T16:45:50.015024Z  INFO Epoch 10   | Score:   1.6821 | Time: 1.13ms
2026-11-15T16:45:50.015097Z  INFO Epoch 11   | Score:   1.6821 | Time: 1.19ms
2026-11-15T16:45:50.015154Z  INFO Epoch 12   | Score:   1.68

In [3]:
collector.plot("scores.best", "pct.diversity")

In [4]:
eval_results = result.value.eval(inputs)
accuracy = rd.accuracy(result.value, inputs, answers, loss=rd.MSE)
accuracy

PyAccuracy("Regression Accuracy" {
	N: 20 
	Accuracy: 99.88%
	R² Score: 0.99994
	RMSE: 0.01610
	Loss (MSE): 0.00026
})

In [5]:
df = collector.to_polars(lazy=False)
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",9.0,109.0,54.5,64.346718,4140.5,NaN,9.0,100.0,2,null,null,null,null,null,null,0,2,"[""statistic""]"
"""step.evaluate.time""",0.000076,0.000376,0.000188,0.000159,2.5256e-8,NaN,0.000076,0.000301,2,376µs,188µs,158µs,75µs,300µs,0µs,0,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,20.0,20.0,0.0,0.0,NaN,20.0,20.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000048,0.000048,0.000048,0.0,0.0,NaN,0.000048,0.000048,1,47µs,47µs,0µs,47µs,47µs,0µs,0,1,"[""selector"", ""time""]"
"""selector.tournament""",80.0,80.0,80.0,0.0,0.0,NaN,80.0,80.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""crossover.graph.rate.parent""",0.5,128.0,0.5,0.0,0.0,0.0,0.5,0.5,256,null,null,null,null,null,null,255,1,"[""alterer"", ""crossover"", … ""expr""]"
"""step.metric.time""",0.000008,0.002334,0.000009,0.000005,2.4345e-11,0.0,0.000007,0.000082,256,2334µs,9µs,4µs,7µs,82µs,0µs,255,1,"[""time"", ""step""]"
"""time""",0.000122,0.021336,0.000083,0.000048,2.3305e-9,0.0,0.000036,0.000698,256,21336µs,83µs,48µs,36µs,698µs,0µs,255,1,"[""time""]"


In [6]:
filtered = (
    df.filter(pl.col("name") == "score.improvement")
    .select("generation")
    .unique()
    .sort("generation")
)
filtered

generation
i64
0
1
2
3
4
…
251
252
253


In [7]:
display(HTML(filtered._repr_html_()))

generation
i64
0
1
2
3
4
…
251
252
253
